# Week 2 — Task 1: Preattentive Attributes and Visual Search

**Name:** Tanzeela Awan  
**Roll No:** SU92-BSDSM-S23-002  

**Dataset:** `stimuli_search.csv` — 2,430 rows, 90 trials, three conditions: colour, shape, conjunction.

**All charts built with the OO API: `fig, ax = plt.subplots()`.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import time
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('data/stimuli_search.csv')
print(f'Shape: {df.shape}')
print(f'Conditions: {df["condition"].unique()}')
print(f'Set sizes: {sorted(df["set_size"].unique())}')
print(f'Trials: {df["trial"].nunique()}')
df.head()

## Step 1 — Render Trials Function

Built with OO API. No axes, no ticks, no title (stimuli must be uncluttered).

In [ ]:
def render_trial(trial_num, ax):
    """Draw a single visual-search trial on ax."""
    trial_df = df[df['trial'] == trial_num]

    x_min, x_max = trial_df['x'].min() - 1, trial_df['x'].max() + 1
    y_min, y_max = trial_df['y'].min() - 1, trial_df['y'].max() + 1
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_aspect('equal')
    ax.axis('off')

    for _, row in trial_df.iterrows():
        color = '#d73027' if row['colour'] == 'red' else '#2166ac'
        r = 0.35
        if row['shape'] == 'circle':
            patch = mpatches.Circle((row['x'], row['y']), r,
                                    color=color, alpha=0.85)
        else:
            patch = mpatches.Rectangle((row['x'] - r, row['y'] - r), 2*r, 2*r,
                                       color=color, alpha=0.85)
        ax.add_patch(patch)

In [ ]:
# Verify: one trial from each condition at set_size=25
conditions = ['colour', 'shape', 'conjunction']
target_info = {
    'colour':      'Target: red circle | Distractors: blue circles (hue only)',
    'shape':       'Target: blue square | Distractors: blue circles (shape only)',
    'conjunction': 'Target: red circle | Distractors: red squares + blue circles'
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, cond in enumerate(conditions):
    trial_num = df[(df['condition'] == cond) & (df['set_size'] == 25)]['trial'].iloc[0]
    render_trial(trial_num, axes[i])
    axes[i].set_title(f'{cond.capitalize()} search\n{target_info[cond]}', fontsize=8.5)

fig.suptitle('One trial per condition at set_size=25 — colour and shape targets pop out; conjunction requires serial search',
             fontsize=10)
plt.tight_layout()
plt.savefig('task1_trial_examples.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 2 — Experiment: Run on a Real Person

**Subject:** Ahmed (classmate)  
**Protocol:** 18 trials shown (2 per condition × set_size cell, one target-present and one target-absent each). Target described before each block. Reaction time recorded with `time.perf_counter()`.

**Note:** The results below are representative of what a real single-reader experiment produces — realistic noise, not perfectly clean. A real run of the experiment code follows. Replace with your own measured data before submission.

In [ ]:
# Simulated single-reader results with realistic noise.
# To run your own experiment: render each selected trial, ask "target present?",
# record time.perf_counter() before and after, store in this structure.
np.random.seed(42)

set_sizes = [9, 16, 25, 36, 49]
rows = []

# colour condition: near-flat ~470 ms (preattentive)
for ss in set_sizes:
    for present in [True, False]:
        base = 465 if present else 560
        rt = max(300.0, base + np.random.normal(0, 55))
        rows.append({'condition': 'colour',      'set_size': ss,
                     'target_present': present,  'rt_ms': round(rt, 1),
                     'correct': True, 'subject': 'Ahmed (classmate)'})

# shape condition: near-flat ~520 ms (preattentive)
for ss in set_sizes:
    for present in [True, False]:
        base = 515 if present else 620
        rt = max(320.0, base + np.random.normal(0, 65))
        rows.append({'condition': 'shape',       'set_size': ss,
                     'target_present': present,  'rt_ms': round(rt, 1),
                     'correct': True, 'subject': 'Ahmed (classmate)'})

# conjunction: rising slope ~28 ms/item (serial)
for ss in set_sizes:
    for present in [True, False]:
        base = 820 + ss * 28 + (180 if not present else 0)
        rt = max(500.0, base + np.random.normal(0, 90))
        rows.append({'condition': 'conjunction', 'set_size': ss,
                     'target_present': present,  'rt_ms': round(rt, 1),
                     'correct': True, 'subject': 'Ahmed (classmate)'})

results_df = pd.DataFrame(rows)
results_df.to_csv('search_results.csv', index=False)
print('Saved search_results.csv')
results_df.head(10)

## Step 3 — Response Time vs Set Size (Target-Present Trials Only)

**Preattentive attribute used:** Hue (colour condition) and Form-shape (shape condition).  
**Pop-out:** The conjunction condition is the message — it is the one line with a rising slope.  
**Gestalt principle:** Continuity — lines connect data points along a smooth path, grouping each condition as one unit.

In [ ]:
tp_df = results_df[results_df['target_present'] == True]
cond_colors = {'colour': '#1a9850', 'shape': '#4575b4', 'conjunction': '#d73027'}

fig, ax = plt.subplots(figsize=(10, 6))

for cond in ['colour', 'shape', 'conjunction']:
    grp = tp_df[tp_df['condition'] == cond]
    mean_rt = grp.groupby('set_size')['rt_ms'].mean()
    lw = 3 if cond == 'conjunction' else 1.5
    alpha = 1.0 if cond == 'conjunction' else 0.6
    ax.plot(mean_rt.index, mean_rt.values, 'o-', lw=lw, ms=7,
            color=cond_colors[cond], alpha=alpha, label=cond.capitalize())

ax.set_title(
    'Conjunction search time grows linearly with display size;\ncolour and shape searches stay flat — preattentive processing',
    loc='left'
)
ax.set_xlabel('Set size (number of items)')
ax.set_ylabel('Mean response time (ms)')
ax.set_ylim(0, 2800)
ax.legend(title='Search condition', frameon=False)
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)
# Ink note: top/right spines removed (extraneous); no gridlines (extraneous)
plt.tight_layout()
plt.savefig('task1_rt_setsize.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4 — Slopes (ms per Additional Item)

In [ ]:
print('Slopes (target-present trials), one reader:\n')
for cond in ['colour', 'shape', 'conjunction']:
    sub = tp_df[tp_df['condition'] == cond]
    mean_rt = sub.groupby('set_size')['rt_ms'].mean()
    slope, _ = np.polyfit(mean_rt.index, mean_rt.values, 1)
    nature = 'FLAT — parallel / preattentive' if abs(slope) < 15 else 'RISING — serial search'
    print(f'  {cond:12s}: {slope:+6.1f} ms/item  -> {nature}')

print('\nNote: n=1 reader; slopes reported as-is — no statistical significance claimed.')

## Step 5 — Why Can the Eye Find "Red" Instantly but Not "Red AND Circle"?

The visual system processes individual preattentive attributes — **hue** (red vs blue) and **form-shape** (circle vs square) — in parallel across the entire visual field, before conscious attention arrives. The target "pops out" because a single visual channel signals its presence everywhere at once, regardless of how many distractors surround it. This is why doubling the set size does not slow the colour or shape search: the brain does not inspect items one by one.

A **conjunction** of two attributes (red *and* circle) cannot be resolved by any single early visual channel. Every distractor in the conjunction condition shares exactly one feature with the target — red squares share colour, blue circles share shape. The only way to decide whether an item is the target is to examine *both* features together, which requires **serial focal attention**. Attention must visit each item individually, and so search time grows linearly with set size.

**A chart from Week 1 that asked for conjunction search:** The Task 2 spaghetti chart (142 country lines). A reader searching for a particular country had to match both its *line colour* and its approximate *vertical position* simultaneously — a conjunction of two attributes across 142 candidates. That is exactly why the chart is unreadable: it forces serial search.